In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

# Importing Data

In [2]:
# reading excel files

paid_subscriptions_df = pd.read_excel('data/paid_subscriptions.xlsx')
events_df = pd.read_excel('data/events.xlsx')

In [3]:
paid_subscriptions_df.head()

,subscription_id,user_id,purchase_timestamp,subscription_type,currency,price
0,0,0,2024-03-04 04:26:39,monthly,EUR,93.78
1,1,1,2024-02-17 11:22:25,quarterly,EUR,62.66
2,2,2,2024-03-03 09:22:45,annual,EUR,71.68
3,3,3,2024-04-11 05:52:34,quarterly,EUR,7.16
4,4,4,2024-02-03 03:46:56,monthly,EUR,9.18


In [4]:
paid_subscriptions_df.dtypes

subscription_id                int64
user_id                        int64
purchase_timestamp    datetime64[ns]
subscription_type             object
currency                      object
price                        float64
dtype: object

In [5]:
paid_subscriptions_df.shape

(500, 6)

In [6]:
events_df.head()

,event_id,user_id,event_timestamp,event_name,feature
0,0,0,2024-02-16 08:03:00,complete_target_practice,Target Practice
1,1,0,2024-02-19 15:16:54,complete_5th_mistake,Unlimited hearts
2,2,0,2024-02-03 04:57:54,complete_review_mistakes,Review Mistakes
3,3,0,2024-02-09 11:47:41,complete_review_mistakes,Review Mistakes
4,4,0,2024-01-27 22:47:53,complete_use_ai_chatbot,AI Chatbot


In [7]:
events_df.dtypes

event_id                    int64
user_id                     int64
event_timestamp    datetime64[ns]
event_name                 object
feature                    object
dtype: object

In [8]:
events_df.shape

(20000, 5)

# Preprocessing Data

In [9]:
attribution_window = 14

In [10]:
# creating all possible pairs of paid subscriptions and the events following them in a defined attribution window

attribution_pairs = (
    pd.merge(left=paid_subscriptions_df, right=events_df, how='left', on=['user_id', 'user_id'])
    .assign(attribution_days = lambda df: (df['purchase_timestamp'] - df['event_timestamp']).dt.days)
    .query('`attribution_days` > 0 & `attribution_days` <= @attribution_window')
    .reset_index(drop=True)
)

attribution_pairs

,subscription_id,user_id,purchase_timestamp,subscription_type,currency,price,event_id,event_timestamp,event_name,feature,attribution_days
0,0,0,2024-03-04 04:26:39,monthly,EUR,93.78,1,2024-02-19 15:16:54,complete_5th_mistake,Unlimited hearts,13
1,0,0,2024-03-04 04:26:39,monthly,EUR,93.78,9,2024-02-26 04:50:26,complete_5th_mistake,Unlimited hearts,6
2,0,0,2024-03-04 04:26:39,monthly,EUR,93.78,14230,2024-02-18 19:15:49,complete_use_ai_chatbot,AI Chatbot,14
3,1,1,2024-02-17 11:22:25,quarterly,EUR,62.66,10,2024-02-07 13:49:28,complete_review_mistakes,Review Mistakes,9
4,1,1,2024-02-17 11:22:25,quarterly,EUR,62.66,11,2024-02-11 20:23:10,complete_use_ai_chatbot,AI Chatbot,5
...,...,...,...,...,...,...,...,...,...,...,...
2363,499,499,2024-01-21 22:48:17,monthly,EUR,34.71,4996,2024-01-07 18:44:19,complete_5th_mistake,Unlimited hearts,14
2364,499,499,2024-01-21 22:48:17,monthly,EUR,34.71,4998,2024-01-20 18:08:04,complete_target_practice,Target Practice,1
2365,499,499,2024-01-21 22:48:17,monthly,EUR,34.71,4999,2024-01-09 17:22:29,complete_target_practice,Target Practice,12
2366,499,499,2024-01-21 22:48:17,monthly,EUR,34.71,5009,2024-01-20 17:27:19,complete_use_ai_chatbot,AI Chatbot,1


# Last Touch Attribution

In [11]:
last_touch = (
    attribution_pairs
    .assign(
        last_touch_rank=(
            attribution_pairs
            .groupby('user_id')
            ['event_timestamp']
            .rank(method='first', ascending=False) # ranking the latest triggered event as 1st
        )
    )
    .query('`last_touch_rank`==1')
    .assign(attributed_feature=lambda df: df['feature'])
    .reset_index(drop=True)
)

last_touch.head()

,subscription_id,user_id,purchase_timestamp,subscription_type,currency,price,event_id,event_timestamp,event_name,feature,attribution_days,last_touch_rank,attributed_feature
0,0,0,2024-03-04 04:26:39,monthly,EUR,93.78,9,2024-02-26 04:50:26,complete_5th_mistake,Unlimited hearts,6,1.0,Unlimited hearts
1,1,1,2024-02-17 11:22:25,quarterly,EUR,62.66,15,2024-02-14 01:21:34,complete_target_practice,Target Practice,3,1.0,Target Practice
2,3,3,2024-04-11 05:52:34,quarterly,EUR,7.16,32,2024-04-07 00:43:28,complete_review_mistakes,Review Mistakes,4,1.0,Review Mistakes
3,4,4,2024-02-03 03:46:56,monthly,EUR,9.18,41,2024-01-29 15:02:44,complete_target_practice,Target Practice,4,1.0,Target Practice
4,5,5,2024-05-03 19:10:03,quarterly,EUR,51.82,10075,2024-04-29 18:15:48,complete_review_mistakes,Review Mistakes,4,1.0,Review Mistakes


# First Touch Attribution

In [12]:
first_touch = (
    attribution_pairs
    .assign(
        last_touch_rank=(
            attribution_pairs
            .groupby('user_id')
            ['event_timestamp']
            .rank(method='first', ascending=True) # ranking the oldest triggered event as 1st
        )
    )
    .query('`last_touch_rank`==1')
    .assign(attributed_feature=lambda df: df['feature'])
    .reset_index(drop=True)
)

first_touch.head()

,subscription_id,user_id,purchase_timestamp,subscription_type,currency,price,event_id,event_timestamp,event_name,feature,attribution_days,last_touch_rank,attributed_feature
0,0,0,2024-03-04 04:26:39,monthly,EUR,93.78,14230,2024-02-18 19:15:49,complete_use_ai_chatbot,AI Chatbot,14,1.0,AI Chatbot
1,1,1,2024-02-17 11:22:25,quarterly,EUR,62.66,10,2024-02-07 13:49:28,complete_review_mistakes,Review Mistakes,9,1.0,Review Mistakes
2,3,3,2024-04-11 05:52:34,quarterly,EUR,7.16,11421,2024-03-27 16:18:30,complete_target_practice,Target Practice,14,1.0,Target Practice
3,4,4,2024-02-03 03:46:56,monthly,EUR,9.18,43,2024-01-19 14:47:19,complete_target_practice,Target Practice,14,1.0,Target Practice
4,5,5,2024-05-03 19:10:03,quarterly,EUR,51.82,5781,2024-04-19 01:05:41,complete_target_practice,Target Practice,14,1.0,Target Practice


# Multi-Touch Attribution (Linear)

In [13]:
# deduplicating events triggers with high frequencies 
latest_event_triggers = (
    attribution_pairs
    .sort_values(by='event_timestamp', ascending=False) # taking the most recent timestamp for each event trigger
    .groupby(['user_id', 'event_name'])
    .nth(0)
    .reset_index(drop=True)
)

# unique features used
feature_usage_count_by_user = (
    latest_event_triggers
    .groupby('user_id')
    .agg(features_used=('event_name', 'nunique'))
    .reset_index()
)

# assigning credit
multi_touch = (
    latest_event_triggers
    .merge(feature_usage_count_by_user, how='left', on='user_id')
    .assign(credit_attributed = lambda df: round(1/df['features_used'], 2))
    .assign(revenue_attributed = lambda df: df['credit_attributed'] * df['price'])
)

multi_touch.head()

,subscription_id,user_id,purchase_timestamp,subscription_type,currency,price,event_id,event_timestamp,event_name,feature,attribution_days,features_used,credit_attributed,revenue_attributed
0,466,466,2024-06-29 04:23:20,annual,EUR,99.37,6561,2024-06-26 09:32:53,complete_5th_mistake,Unlimited hearts,2,2,0.50,49.6850
1,466,466,2024-06-29 04:23:20,annual,EUR,99.37,15062,2024-06-26 05:17:24,complete_review_mistakes,Review Mistakes,2,2,0.50,49.6850
2,490,490,2024-06-29 15:49:20,quarterly,EUR,83.03,9009,2024-06-25 09:58:03,complete_review_mistakes,Review Mistakes,4,4,0.25,20.7575
3,87,87,2024-06-25 19:59:00,annual,EUR,78.40,18146,2024-06-24 05:49:44,complete_target_practice,Target Practice,1,1,1.00,78.4000
4,69,69,2024-06-25 13:02:36,monthly,EUR,62.19,693,2024-06-24 02:38:08,complete_5th_mistake,Unlimited hearts,1,3,0.33,20.5227


# Multi-Touch Attribution (Time-Decay)

In [14]:
half_life = 7

# for a 14-day attribution, we use a half life of 7 days
# it keeps the decay function balanced and captures both early exploration and late conversion behaviors

In [15]:
# deduplicating events triggers with high frequencies 
latest_event_triggers = (
    attribution_pairs
    .sort_values(by='event_timestamp', ascending=False) # taking the most recent timestamp for each event trigger
    .groupby(['user_id', 'event_name'])
    .nth(0)
    .reset_index(drop=True)
)

latest_event_triggers.head()

,subscription_id,user_id,purchase_timestamp,subscription_type,currency,price,event_id,event_timestamp,event_name,feature,attribution_days
0,466,466,2024-06-29 04:23:20,annual,EUR,99.37,6561,2024-06-26 09:32:53,complete_5th_mistake,Unlimited hearts,2
1,466,466,2024-06-29 04:23:20,annual,EUR,99.37,15062,2024-06-26 05:17:24,complete_review_mistakes,Review Mistakes,2
2,490,490,2024-06-29 15:49:20,quarterly,EUR,83.03,9009,2024-06-25 09:58:03,complete_review_mistakes,Review Mistakes,4
3,87,87,2024-06-25 19:59:00,annual,EUR,78.40,18146,2024-06-24 05:49:44,complete_target_practice,Target Practice,1
4,69,69,2024-06-25 13:02:36,monthly,EUR,62.19,693,2024-06-24 02:38:08,complete_5th_mistake,Unlimited hearts,1


In [16]:
# determining weights using half-life formula
latest_event_triggers = (
    latest_event_triggers
    .assign(time_decay_weight=lambda df: 0.5 ** (df['attribution_days'] * (1/half_life)))
)

time_decay_weight_sums = (
    latest_event_triggers
    .groupby('user_id')
    .agg(time_decay_weight_sum=('time_decay_weight', 'sum'))
)

# assigning credit
multi_touch_decay = (
    latest_event_triggers
    .merge(time_decay_weight_sums, how='left', on='user_id')
    .assign(credit_attributed=lambda df: round(df['time_decay_weight']/df['time_decay_weight_sum'], 2))
    .assign(revenue_attributed = lambda df: df['credit_attributed'] * df['price'])
)

multi_touch_decay.head()

,subscription_id,user_id,purchase_timestamp,subscription_type,currency,price,event_id,event_timestamp,event_name,feature,attribution_days,time_decay_weight,time_decay_weight_sum,credit_attributed,revenue_attributed
0,466,466,2024-06-29 04:23:20,annual,EUR,99.37,6561,2024-06-26 09:32:53,complete_5th_mistake,Unlimited hearts,2,0.820335,1.640671,0.50,49.6850
1,466,466,2024-06-29 04:23:20,annual,EUR,99.37,15062,2024-06-26 05:17:24,complete_review_mistakes,Review Mistakes,2,0.820335,1.640671,0.50,49.6850
2,490,490,2024-06-29 15:49:20,quarterly,EUR,83.03,9009,2024-06-25 09:58:03,complete_review_mistakes,Review Mistakes,4,0.672950,1.773333,0.38,31.5514
3,87,87,2024-06-25 19:59:00,annual,EUR,78.40,18146,2024-06-24 05:49:44,complete_target_practice,Target Practice,1,0.905724,0.905724,1.00,78.4000
4,69,69,2024-06-25 13:02:36,monthly,EUR,62.19,693,2024-06-24 02:38:08,complete_5th_mistake,Unlimited hearts,1,0.905724,2.321671,0.39,24.2541


# Attributed Performance

In [18]:
df_dict = {
    'last_touch': last_touch,
    'first_touch': first_touch, 
    'multi_touch': multi_touch, 
    'multi_touch_decay': multi_touch_decay
}

df_performance_dict = {}

for df in df_dict.keys():
    df_performance_dict[f'{df}_performance'] = (
        df_dict[df]
        .assign(attribution_model = f'{df}'.replace('_', ' ').title())
        .groupby(['attribution_model', 'feature'])
        .agg(
            attributed_user_conversions=('user_id', 'nunique'),
            attributed_revenue=('price', 'sum')
        )
        .round({'attributed_user_conversions': 0, 'attributed_revenue': 0})
        .reset_index()
    )

all_attribution_models = pd.concat(df_performance_dict.values())

all_attribution_models

,attribution_model,feature,attributed_user_conversions,attributed_revenue
0,Last Touch,AI Chatbot,133,7163.0
1,Last Touch,Review Mistakes,87,4884.0
2,Last Touch,Target Practice,149,7662.0
3,Last Touch,Unlimited hearts,112,5842.0
0,First Touch,AI Chatbot,144,7816.0
1,First Touch,Review Mistakes,103,5377.0
2,First Touch,Target Practice,139,7176.0
3,First Touch,Unlimited hearts,95,5181.0
0,Multi Touch,AI Chatbot,358,19131.0
1,Multi Touch,Review Mistakes,288,15474.0


In [21]:
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display

def plot_attribution_charts(df, metric):
    attribution_models = df['attribution_model'].unique()
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.flatten()

    for i, model in enumerate(attribution_models):
        ax = axes[i]
        sns.barplot(
            data=df[df['attribution_model'] == model],
            x='feature',
            y=metric,
            ax=axes[i],
            hue='feature',
            palette='Blues'
        )
        ax.set_title(model, fontsize=16)
        ax.set_xlabel('Event Name')
        ax.set_ylabel(metric.replace('_', ' ').title())
        ax.tick_params(axis='x', labelrotation=45)

    plt.tight_layout()
    plt.show()

# Dropdown to choose y-variable
metric_selector = widgets.Dropdown(
    options=[
        ('Attributed User Conversions', 'attributed_user_conversions'),
        ('Attributed Revenue', 'attributed_revenue')
    ],
    value='attributed_revenue',
    description='Select Metric:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='330px')
)

# Use interactive output
output = widgets.interactive_output(
    plot_attribution_charts,
    {'df': widgets.fixed(all_attribution_models), 'metric': metric_selector}
)

display(metric_selector, output)


Dropdown(description='Select Metric:', index=1, layout=Layout(width='330px'), options=(('Attributed User Conve…

Output()